In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Qubit initialization

<Accordion>
<AccordionItem title="เวอร์ชันของแพ็กเกจ">

โค้ดในหน้านี้พัฒนาโดยใช้ requirements ต่อไปนี้
แนะนำให้ใช้เวอร์ชันนี้หรือใหม่กว่า

```
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>

เมื่อ Circuit ถูกประมวลผลบน IBM&reg; quantum processing unit (QPU) โดยทั่วไปจะมีการแทรก implicit reset ที่ต้นของ Circuit เพื่อให้แน่ใจว่า Qubit ถูก initialize เป็นศูนย์ ซึ่งควบคุมโดย flag `init_qubits` ที่กำหนดไว้เป็น [primitive execution option](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2)

อย่างไรก็ตาม ความไม่สมบูรณ์ในกระบวนการ reset อาจนำไปสู่ state-preparation error เพื่อลดความผิดพลาด QPU ยังแทรก repetition delay time (หรือ `rep_delay`) ระหว่าง Circuit ด้วย Backend แต่ละตัวมี `rep_delay` ค่าเริ่มต้นที่แตกต่างกัน แต่ปกติจะถูกกำหนดเพื่อสมดุลระหว่างความแม่นยำของการ reset กับเวลาการประมวลผลรวม รัน `backend.default_rep_delay` เพื่อดูค่า `rep_delay` เริ่มต้นของ QPU ที่ต้องการ

เนื่องจาก IBM QPU ทั้งหมดใช้ dynamic repetition rate execution คุณสามารถเปลี่ยน `rep_delay` สำหรับแต่ละ job ได้ Circuit ที่คุณส่งใน primitive job จะถูกรวมเป็นกลุ่มเพื่อประมวลผลบน QPU Circuit เหล่านี้ถูกประมวลผลโดยการวน iterate ข้าม Circuit สำหรับแต่ละ shot ที่ร้องขอ โดยการประมวลผลเป็นแบบ column-wise ข้ามเมทริกซ์ของ Circuit และ shot ดังแสดงในรูปต่อไปนี้

![คอลัมน์แรกแสดง shot ที่ 0 Circuit ถูกรันตามลำดับจาก 0 ถึง 3 คอลัมน์ที่สองแสดง shot ที่ 1 Circuit ถูกรันตามลำดับจาก 0 ถึง 3 คอลัมน์ที่เหลือทำตามรูปแบบเดิม](../docs/images/guides/repetition-rate-execution/circuits_shots_matrix1.avif "Column-wise execution matrix")

เนื่องจาก `rep_delay` ถูกแทรกระหว่าง Circuit แต่ละ shot ของการประมวลผลจะพบกับ delay นี้ ดังนั้น เมื่อคุณลด `rep_delay` เวลาประมวลผล QPU รวมจะลดลง แต่แลกกับ state preparation error rate ที่เพิ่มขึ้น ดังที่รูปต่อไปนี้แสดง:

![รูปนี้แสดงว่าเมื่อค่า `rep_delay` ลดลง state preparation error rate จะเพิ่มขึ้น](../docs/images/guides/repetition-rate-execution/rep_delay.avif "Repetition delay versus error rate")

ถ้าคุณตั้งค่าทั้ง `rep_delay=0` และ `init_qubits=False` Circuit จะ "รวม" เข้าด้วยกัน เนื่องจาก Qubit จะเริ่มในสถานะสุดท้ายจาก shot ก่อนหน้า

โปรดทราบว่าแม้ Circuit ใน primitive job จะถูกรวมเป็นกลุ่มสำหรับการประมวลผล QPU แต่ไม่มีการรับประกันลำดับที่ Circuit จาก PUB จะถูกประมวลผล ตัวอย่างเช่น ถ้าคุณส่ง `pubs=[pub1, pub2]` Circuit จาก `pub1` อาจไม่รันก่อน `pub2` และไม่มีการรับประกันว่า Circuit จาก job เดียวกันจะรันเป็น batch เดียวบน QPU

## ระบุ `rep_delay` สำหรับ primitive job
> **Note:** ตรวจสอบช่วง `rep_delay` ที่รองรับสำหรับ QPU ที่คุณใช้งานเสมอ ค่าเหล่านี้ไม่เหมือนกันสำหรับทุก QPU และอาจเปลี่ยนแปลงได้ตามเวลา
> 
> โปรดทราบว่าการเพิ่ม `rep_delay` จะส่งผลโดยตรงต่อเวลาการประมวลผลและการใช้งาน capacity ของคุณ

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService()

# Make sure your backend supports it
backend = service.least_busy(
    operational=True, min_num_qubits=100, dynamic_reprate_enabled=True
)

# Determine the allowable range
backend.rep_delay_range
sampler = Sampler(mode=backend)

# Specify a value in the supported range
sampler.options.execution.rep_delay = 0.0005

## ขั้นตอนถัดไป
> **Tip:** - ลองทำตัวอย่างใน tutorial [Quantum approximate optimization algorithm (QAOA)](/tutorials/quantum-approximate-optimization-algorithm)
>   - ทบทวนวิธี [เริ่มต้นใช้งาน Estimator](/guides/get-started-with-estimator)
>   - ทบทวนวิธี [เริ่มต้นใช้งาน Sampler](/guides/get-started-with-sampler)